In [3]:
# 1. Install python3.10 and venv support
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-venv python3.10-dev -y
# 2. Create an isolated virtual environment
!python3.10 -m venv /content/venv
# 3. Upgrade pip inside the virtual environment
!/content/venv/bin/python -m pip install --upgrade pip
# 4. Install the required serving pins
!/content/venv/bin/python -m pip install \
    "vllm==0.6.*" \
    "transformers==4.46.*" \
    "accelerate==1.1.*" \
    "httpx==0.27.*" \
    "openai==1.54.*"
print("Virtual environment ready with vLLM installed!")

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,915 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,180 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,316 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-drivers/ppa/u

In [6]:
# %% [markdown]
# # Week 3 shared Colab scaffold
#
# This file is the source of truth for the reusable Colab cells every week-3 lab
# uses. It is written in py-percent format: each `# %%` block is one standalone,
# pasteable Colab cell. Copy the cells you need into the day's notebook in the
# order the lab README gives.
#
# Why this scaffold exists: a Colab notebook runs cells one at a time, top to
# bottom, and a cell blocks until it returns. A live inference server does not
# return: it runs until you kill it. So you cannot "start the server" in one cell
# and "watch it" in the next the way you would in a terminal with two panes. The
# pattern here is launch-then-poll: one cell launches the server as a background
# subprocess and returns immediately, and a second cell polls the health endpoint
# until the server answers or a timeout fires. Every long-running piece (the
# server, the nvidia-smi sampler) runs in the background and is watched by a
# short cell that returns.
#
# Pin source: versions come from ../../../PINS.md (course root). The vLLM-on-T4 pin
# is verified on a real free-tier T4 before the cohort starts; the confirmed
# version and date land in PINS.md under "Verification status". Do not invent a
# vLLM version here; read the pin.
#
# Convert to a .ipynb when you want a notebook file (the .py stays the source of
# truth):
#   uvx jupytext --to ipynb colab_scaffold.py

# %%
# PINS block. These mirror ../../../PINS.md (course root, the single source of
# truth). If a pin changes, it changes in PINS.md first, then here. The vLLM pin
# is the load-bearing one: it must be the version confirmed on a real free-tier
# T4 during the pre-cohort verification pass. Read PINS.md before you run this.
#
# PINS (from ../../../PINS.md):
#   VLLM_PIN=0.6.*          # OpenAI server; runs the xformers backend on sm75
#   BITSANDBYTES_PIN=0.49.2 # int8/int4 load path (day 1 profiling); 0.44.* is
#                           # broken on Colab's cu128 torch, see PINS.md
#   AUTOAWQ_PIN=0.2.*       # AWQ weights load path (day 4)
#   TRANSFORMERS_PIN=4.46.* # streaming generation (day 2)
#   ACCELERATE_PIN=1.1.*    # device placement
#   HTTPX_PIN=0.27.*        # async A/B client (day 3)
#   OPENAI_PIN=1.54.*       # the client that proves the /v1 contract

# %%
# Cell: the pins and the installer function. Defines only, installs nothing.
# Paste this on every week-3 day. Then paste ONE of the two install cells below,
# whichever the day's README names. Day 1 profiles with transformers and must
# NOT install vLLM; days 2 to 5 serve, and must.
import subprocess, sys

# Pins mirrored from ../../../PINS.md. Keep these two in sync (PINS.md wins).
VLLM_PIN = "0.6.*"
BITSANDBYTES_PIN = "0.49.2"
AUTOAWQ_PIN = "0.2.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

def pip_install(*specs):
    cmd = ["/content/venv/bin/python", "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

# %%
# INSTALL CELL A: profiling only, NO SERVER. This is day 1.
#
# Day 1 loads the model with transformers and reads the card. It never serves,
# so it must NOT install vLLM. Installing vLLM here would replace Colab's torch
# with vLLM's older build AND downgrade numpy to 1.26, and Colab's preinstalled
# extensions are compiled against numpy 2. The model load then dies with
#   RuntimeError: Failed to import transformers.models.qwen2.modeling_qwen2
#   ... numpy.dtype size changed, Expected 96 from C header, got 88
# Verified on a T4, 2026-07-27. Colab's own torch is the torch today. This
# install is about two minutes, not thirty.
pip_install(
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"bitsandbytes=={BITSANDBYTES_PIN}",
)
print("profiling pins installed (no vLLM today)")

# %%
# INSTALL CELL B: the serving set. This is days 3 to 5 (day 2 is CELL A:
# direct transformers loads crash on vLLM's numpy - verified on T4 2026-08-07). About 30 minutes on a
# cold runtime, and it prints almost nothing for most of it, so start it and go
# and fill in your prediction card. vLLM brings its own torch; do NOT install a
# second one.
#
# transformers and accelerate are NOT optional here, even on days you never call
# them directly. vLLM 0.6.x installs its own torch (2.5.1), which downgrades
# Colab's torch and leaves Colab's preinstalled torchaudio compiled against the
# wrong ABI. Colab's preinstalled transformers imports torchaudio at module load,
# so vLLM then dies during startup with
#   OSError: _torchaudio.abi3.so: undefined symbol: aoti_torch_abi_version
# Pinning transformers to 4.46 removes that import path. Verified on a T4,
# 2026-07-27: without these two lines the server never comes up.
#
# autoawq is only needed on day 4; that README says so and adds it to this call.
pip_install(
    f"vllm=={VLLM_PIN}",
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"httpx=={HTTPX_PIN}",
    f"openai=={OPENAI_PIN}",
)
# NOTE (2026-08-07, verified the hard way on a live T4): do NOT add a
# numpy>=2 pin here - vLLM 0.6.x requires numpy<2 and the install fails
# outright. CELL B as verified 2026-07-27 runs on the numpy vLLM chooses.
print("serving pins installed")

# %%
# Cell: launch the server as a background subprocess.
# vLLM's OpenAI-compatible server runs until killed, so it cannot live in a cell
# that must return. Popen launches it in the background and this cell returns at
# once. stdout and stderr are teed to /content/server.log so the health-poll cell
# and you can read what happened. Flags come from PINS.md canon: --dtype half is
# mandatory on the T4 (sm75 has no bf16 and no FlashAttention, so vLLM uses the
# xformers backend). Edit SERVER_ARGS for the day (day 3 is plain, day 4 adds
# --quantization awq and the tool-call flags).
import os, signal, subprocess

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 8000
SERVER_LOG = "/content/server.log"

# Args as a dict so a lab can override one value without retyping the line.
SERVER_ARGS = {
    "--model": MODEL,
    "--dtype": "half",                 # sm75: no bf16, no FlashAttention
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(PORT),
}

def build_cmd(args: dict) -> list:
    cmd = ["/content/venv/bin/python", "-m", "vllm.entrypoints.openai.api_server"]
    for k, v in args.items():
        if v is None:            # bare flag, e.g. "--enable-auto-tool-choice": None
            cmd.append(k)
        else:
            cmd += [k, str(v)]
    return cmd

def launch_server(args: dict = None):
    args = SERVER_ARGS if args is None else args
    cmd = build_cmd(args)
    print("launching:", " ".join(cmd))
    logf = open(SERVER_LOG, "wb")
    # start_new_session=True puts the server in its own process group so the
    # shutdown cell can kill the whole group, not just the parent pid.
    proc = subprocess.Popen(
        cmd, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True,
    )
    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server()

# %%
# Cell: health poll.
# The launch cell returned immediately; the server is still loading weights in
# the background. This cell polls GET /v1/models until it answers 200 or the
# timeout fires. First launch on a fresh runtime downloads the model, so the
# first poll can take a while; that is what the 300s timeout is for. On timeout
# it prints the last 30 log lines so you can see why (usually still downloading,
# or an OOM, or a bad flag).
import time, urllib.request, urllib.error

def tail_log(path=SERVER_LOG, n=30):
    try:
        with open(path, "r", errors="replace") as fh:
            lines = fh.readlines()
        return "".join(lines[-n:])
    except FileNotFoundError:
        return "(no log file yet)"

def wait_for_health(port=PORT, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(f"server healthy after about {waited}s: {url} -> 200")
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass  # not up yet
        time.sleep(interval_s)
    print(f"TIMED OUT after {timeout_s}s waiting for {url}")
    print("last 30 log lines:")
    print(tail_log())
    print("server did not come up. common causes: model still downloading "
          "(rerun this cell), OOM at load (lower --gpu-memory-utilization to "
          "0.80), or a bad flag (bf16 on sm75; use --dtype half).")
    return False

healthy = wait_for_health()

# %%
# Cell: nvidia-smi sampler thread.
# A daemon thread samples GPU utilisation and memory every 2s into a CSV. It is a
# thread, not a subprocess, so it stops when the runtime does and never outlives
# the notebook. Start it before a measurement, stop it after. Do NOT start it
# twice: two samplers write interleaved rows and double your entries (a named
# failure mode in the day-1 lab). start_sampler() guards against that.
import csv, threading, time

GPU_SAMPLES = "/content/gpu_samples.csv"
_sampler = {"thread": None, "stop": None}

def _sample_loop(stop_event, path, interval_s):
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["t", "util_gpu", "mem_used_mib"])
        t0 = time.time()
        while not stop_event.is_set():
            out = subprocess.run(
                ["nvidia-smi",
                 "--query-gpu=utilization.gpu,memory.used",
                 "--format=csv,noheader,nounits"],
                capture_output=True, text=True,
            ).stdout.strip()
            # e.g. "37, 4210"
            parts = [p.strip() for p in out.split(",")]
            if len(parts) == 2:
                w.writerow([round(time.time() - t0, 2), parts[0], parts[1]])
                fh.flush()
            stop_event.wait(interval_s)

def start_sampler(path=GPU_SAMPLES, interval_s=2):
    if _sampler["thread"] and _sampler["thread"].is_alive():
        print("sampler already running; not starting a second one")
        return
    stop = threading.Event()
    th = threading.Thread(
        target=_sample_loop, args=(stop, path, interval_s), daemon=True,
    )
    th.start()
    _sampler["thread"], _sampler["stop"] = th, stop
    print(f"sampler started -> {path} (every {interval_s}s)")

def stop_sampler():
    if _sampler["stop"]:
        _sampler["stop"].set()
    if _sampler["thread"]:
        _sampler["thread"].join(timeout=5)
    _sampler["thread"], _sampler["stop"] = None, None
    print("sampler stopped")

def read_util_mean(path=GPU_SAMPLES):
    """Mean GPU utilisation over the samples on file. Use it after stop_sampler."""
    vals = []
    with open(path) as fh:
        for row in csv.DictReader(fh):
            try:
                vals.append(float(row["util_gpu"]))
            except (KeyError, ValueError):
                pass
    return sum(vals) / len(vals) if vals else 0.0

# %%
# Cell: clean shutdown.
# Terminate the server process group and confirm port 8000 is free again. Run
# this between labs, or before relaunching with different flags. Killing only the
# parent pid can leave a child holding the port; killpg kills the whole group the
# launch cell created with start_new_session=True.
def shutdown_server(proc=None, port=PORT):
    try:
        proc = server if proc is None else proc
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        print(f"sent SIGTERM to process group of pid {proc.pid}")
    except (ProcessLookupError, NameError):
        print("no server process to kill")
    # give it a moment, then confirm the port is free
    time.sleep(3)
    try:
        with urllib.request.urlopen(f"http://localhost:{port}/v1/models", timeout=2):
            print(f"WARNING: port {port} still answering; something is still up")
    except (urllib.error.URLError, ConnectionError, OSError):
        print(f"port {port} is free")

# shutdown_server()   # uncomment to run

# %% [markdown]
# ## RECOVERY: the session-died cell
#
# Free Colab drops runtimes without warning: you lose the GPU, the installed
# packages, and any running server. When that happens you do not re-run the whole
# notebook. Run the ONE cell below. It kills anything left over, reinstalls the
# pins, relaunches the server, and re-polls health. When it prints the healthy
# line you continue from the last step you had finished. Every week-3 lab points
# at this cell at the top for exactly this reason.

# %%
# RECOVERY CELL (self-contained). Runtime died? Run only this cell, then continue
# from your last completed step. It repeats the install, launch, and health-poll
# so you do not have to scroll. Nothing here depends on earlier cells having run.
import os, sys, time, signal, subprocess, urllib.request, urllib.error

RECOVERY_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
RECOVERY_PORT = 8000
RECOVERY_LOG = "/content/server.log"

# Pins mirrored from ../../../PINS.md (keep in sync; PINS.md wins).
_R_TRANSFORMERS = "4.46.*"   # PINS.md: mandatory beside vLLM, every fresh runtime
_R_ACCELERATE = "1.1.*"
_R_NEED_AWQ = False          # set True on day 4+ if your locked model is AWQ
_R_VLLM = "0.6.*"; _R_HTTPX = "0.27.*"; _R_OPENAI = "1.54.*"

# If a day added --quantization awq or the tool-call flags, add them here too so
# recovery brings the server back the way the lab needs it. Default is the plain
# serving config from canon.
RECOVERY_ARGS = {
    "--model": RECOVERY_MODEL,
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(RECOVERY_PORT),
}

# 1) kill any leftover server holding the port
subprocess.run(["pkill", "-f", "vllm.entrypoints.openai.api_server"],
               check=False)
time.sleep(2)

# 2) reinstall pins (fresh runtime has nothing)
subprocess.run(["/content/venv/bin/python", "-m", "pip", "install", "-q",
                f"vllm=={_R_VLLM}", f"transformers=={_R_TRANSFORMERS}",
                f"accelerate=={_R_ACCELERATE}", f"httpx=={_R_HTTPX}",
                f"openai=={_R_OPENAI}"]
               + (["autoawq==0.2.9"] if _R_NEED_AWQ else []),
               check=True)
print("pins reinstalled")

# 3) relaunch the server in the background
_r_cmd = ["/content/venv/bin/python", "-m", "vllm.entrypoints.openai.api_server"]
for k, v in RECOVERY_ARGS.items():
    _r_cmd += [k] if v is None else [k, str(v)]
_r_logf = open(RECOVERY_LOG, "wb")
server = subprocess.Popen(_r_cmd, stdout=_r_logf, stderr=subprocess.STDOUT,
                          start_new_session=True)
print(f"relaunched server pid {server.pid}, logging to {RECOVERY_LOG}")

# 4) re-poll health (first load re-downloads the model; hence 300s)
_deadline = time.time() + 300
while time.time() < _deadline:
    try:
        with urllib.request.urlopen(
                f"http://localhost:{RECOVERY_PORT}/v1/models", timeout=5) as r:
            if r.status == 200:
                print("RECOVERED: server healthy. continue from your last step.")
                break
    except (urllib.error.URLError, ConnectionError, OSError):
        pass
    time.sleep(3)
else:
    print("recovery timed out. last 30 log lines:")
    try:
        with open(RECOVERY_LOG, errors="replace") as fh:
            print("".join(fh.readlines()[-30:]))
    except FileNotFoundError:
        print("(no log file)")
    print("if it keeps timing out: switch to the Kaggle fallback in the shared "
          "README, or rotate to another team Colab account.")

installing: transformers==4.46.* accelerate==1.1.* bitsandbytes==0.49.2
profiling pins installed (no vLLM today)
installing: vllm==0.6.* transformers==4.46.* accelerate==1.1.* httpx==0.27.* openai==1.54.*
serving pins installed
launching: /content/venv/bin/python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000
server pid 18697, logging to /content/server.log
server healthy after about 102s: http://localhost:8000/v1/models -> 200
pins reinstalled
relaunched server pid 19296, logging to /content/server.log
RECOVERED: server healthy. continue from your last step.


In [7]:
import os, sys, time, signal, subprocess, urllib.request, urllib.error

RECOVERY_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
RECOVERY_PORT = 8000
RECOVERY_LOG = "/content/server.log"

_R_TRANSFORMERS = "4.46.*"
_R_ACCELERATE = "1.1.*"
_R_NEED_AWQ = False
_R_VLLM = "0.6.*"; _R_HTTPX = "0.27.*"; _R_OPENAI = "1.54.*"

RECOVERY_ARGS = {
    "--model": RECOVERY_MODEL,
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(RECOVERY_PORT),
}

subprocess.run(["pkill", "-f", "vllm.entrypoints.openai.api_server"], check=False)
time.sleep(2)

subprocess.run(["/content/venv/bin/python", "-m", "pip", "install", "-q",
                f"vllm=={_R_VLLM}", f"transformers=={_R_TRANSFORMERS}",
                f"accelerate=={_R_ACCELERATE}", f"httpx=={_R_HTTPX}",
                f"openai=={_R_OPENAI}"], check=True)

_r_cmd = ["/content/venv/bin/python", "-m", "vllm.entrypoints.openai.api_server"]
for k, v in RECOVERY_ARGS.items():
    _r_cmd += [k] if v is None else [k, str(v)]
_r_logf = open(RECOVERY_LOG, "wb")
server = subprocess.Popen(_r_cmd, stdout=_r_logf, stderr=subprocess.STDOUT, start_new_session=True)

_deadline = time.time() + 300
while time.time() < _deadline:
    try:
        with urllib.request.urlopen(f"http://localhost:{RECOVERY_PORT}/v1/models", timeout=5) as r:
            if r.status == 200:
                print("RECOVERED: server healthy. continue from your last step.")
                break
    except (urllib.error.URLError, ConnectionError, OSError):
        pass
    time.sleep(3)

RECOVERED: server healthy. continue from your last step.


In [8]:
import time, urllib.request, urllib.error

def tail_log(path=SERVER_LOG, n=30):
    try:
        with open(path, "r", errors="replace") as fh:
            lines = fh.readlines()
        return "".join(lines[-n:])
    except FileNotFoundError:
        return "(no log file yet)"

def wait_for_health(port=PORT, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(f"server healthy after about {waited}s: {url} -> 200")
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass
        time.sleep(interval_s)
    print(f"TIMED OUT after {timeout_s}s waiting for {url}")
    print("last 30 log lines:")
    print(tail_log())
    return False

healthy = wait_for_health()

server healthy after about 0s: http://localhost:8000/v1/models -> 200


In [9]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
r = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    messages=[{"role": "user", "content": "In one sentence, what is a GPU?"}],
)
print(r.choices[0].message.content)

A GPU, or Graphics Processing Unit, is a specialized processor designed to accelerate computations involved in rendering graphics and video content on electronic devices.


In [10]:
import json

sample_data = {
    "_note": "SAMPLE baseline from the reference T4 run 2026-08-31.",
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "dtype": "fp16",
    "ttft_s": {
        "128": 0.0371,
        "512": 0.0647,
        "2048": 0.3123
    },
    "tpot_s": 0.0341,
    "batch": {
        "1": 34.0,
        "4": 49.8,
        "8": 96.6
    }
}

with open("baselines.sample.json", "w") as f:
    json.dump(sample_data, f, indent=2)

print("baselines.sample.json created successfully!")

baselines.sample.json created successfully!


In [11]:
raw_code = '''
import asyncio
import time
import httpx

FIXED_PROMPTS = [
    "In one sentence, what is a GPU?",
    "List three reasons decode is memory-bound.",
    "Explain the KV cache to a new ops engineer in two sentences.",
    "What does continuous batching change versus static batching?",
    "Give a one-line definition of tokens per second.",
    "Why does a longer prompt increase time to first token?",
    "Name two things quantisation trades away for smaller memory.",
    "Summarise what an inference server does in three short bullets.",
]

QUEUE = [32, 32, 32, 256] * 6
MAX_TOKENS = 128
WARMUP = 4

async def _one_request(client, base_url, model, prompt, max_tokens=MAX_TOKENS):
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": 0.0,
        "stream": False,
    }
    r = await client.post(f"{base_url}/chat/completions", json=payload)
    r.raise_for_status()
    body = r.json()
    usage = body.get("usage", {})
    ct = usage.get("completion_tokens")
    if ct is None:
        ct = len(body["choices"][0]["message"]["content"].split())
    return ct

async def _run_level(client, base_url, model, prompts, concurrency, total_requests):
    sem = asyncio.Semaphore(concurrency)
    counts = []

    async def guarded(prompt, max_tokens):
        async with sem:
            return await _one_request(client, base_url, model, prompt, max_tokens)

    tasks = [asyncio.create_task(guarded(prompts[i % len(prompts)],
                                        QUEUE[i % len(QUEUE)]))
             for i in range(total_requests)]
    t0 = time.time()
    for coro in asyncio.as_completed(tasks):
        counts.append(await coro)
    dt = time.time() - t0
    total_tokens = sum(counts)
    return {
        "concurrency": concurrency,
        "requests": total_requests,
        "tokens_per_s": round(total_tokens / dt, 1),
        "wall_s": round(dt, 3),
    }

async def run_sweep(base_url, model, prompts=FIXED_PROMPTS,
                    concurrencies=(1, 4, 8), requests_per_level=24):
    results = []
    async with httpx.AsyncClient(timeout=120.0) as client:
        await asyncio.gather(*[
            _one_request(client, base_url, model, prompts[i % len(prompts)])
            for i in range(WARMUP)
        ])
        for c in concurrencies:
            level = await _run_level(client, base_url, model, prompts, c,
                                     requests_per_level)
            print("level:", level)
            results.append(level)
    return results
'''

# Clean non-breaking spaces and save
cleaned_code = raw_code.replace('\xa0', ' ')
with open("ab_client.py", "w") as f:
    f.write(cleaned_code)

# Execute the sweep
from ab_client import run_sweep, FIXED_PROMPTS

vllm_measured = await run_sweep(
    base_url="http://localhost:8000/v1",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    prompts=FIXED_PROMPTS,
    concurrencies=[1, 4, 8],
)

print("Sweep completed successfully!")

level: {'concurrency': 1, 'requests': 24, 'tokens_per_s': 60.5, 'wall_s': 22.951}
level: {'concurrency': 4, 'requests': 24, 'tokens_per_s': 166.3, 'wall_s': 8.352}
level: {'concurrency': 8, 'requests': 24, 'tokens_per_s': 241.4, 'wall_s': 5.754}
Sweep completed successfully!


In [12]:
import json

# Load baseline data
baseline = json.load(open("baselines.sample.json"))

# Map measured throughput and baseline numbers by concurrency level
vllm_by_c = {x["concurrency"]: x["tokens_per_s"] for x in vllm_measured}
base_by_c = {int(k): v for k, v in baseline["batch"].items()}

# Calculate speedup ratios
speedup = {c: round(vllm_by_c[c] / base_by_c[c], 2)
           for c in vllm_by_c if c in base_by_c}

report = {
    "baseline": base_by_c,
    "vllm": vllm_by_c,
    "speedup_by_concurrency": speedup,
    "predicted_speedup": 1.5,
}

# Write out report
with open("ab_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("Generated ab_report.json successfully:\n")
print(json.dumps(report, indent=2))

Generated ab_report.json successfully:

{
  "baseline": {
    "1": 34.0,
    "4": 49.8,
    "8": 96.6
  },
  "vllm": {
    "1": 60.5,
    "4": 166.3,
    "8": 241.4
  },
  "speedup_by_concurrency": {
    "1": 1.78,
    "4": 3.34,
    "8": 2.5
  },
  "predicted_speedup": 1.5
}


In [14]:
static_scaling = base_by_c[8] / base_by_c[1]
vllm_scaling   = vllm_by_c[8] / vllm_by_c[1]

print(f"static batching scales {static_scaling:.2f}x, vLLM scales {vllm_scaling:.2f}x")
print(f"continuous batching is worth {vllm_scaling / static_scaling:.2f}x of scaling")

static batching scales 2.84x, vLLM scales 3.99x
continuous batching is worth 1.40x of scaling


In [15]:
import json
import os

def verify_report():
    report_path = "ab_report.json"
    assert os.path.exists(report_path), f"Error: {report_path} not found!"

    with open(report_path, "r") as f:
        report = json.load(f)

    # 1. Check schema / required keys
    required_keys = ["baseline", "vllm", "speedup_by_concurrency", "predicted_speedup"]
    for k in required_keys:
        assert k in report, f"Schema Error: Missing key '{k}' in ab_report.json"

    # 2. Check that vLLM concurrency-8 throughput beats Monday's batch-8 baseline
    vllm_8 = report["vllm"].get("8") or report["vllm"].get(8)
    base_8 = report["baseline"].get("8") or report["baseline"].get(8)

    assert vllm_8 is not None and base_8 is not None, "Concurrency 8 metrics missing from report."
    assert vllm_8 > base_8, f"Performance check failed: vLLM concurrency-8 ({vllm_8}) must exceed baseline ({base_8})."

    # 3. Verify speedup fields were computed
    speedups = report["speedup_by_concurrency"]
    assert len(speedups) > 0, "Speedup fields were not computed."
    for level, ratio in speedups.items():
        assert isinstance(ratio, (int, float)) and ratio > 0, f"Invalid speedup ratio for level {level}: {ratio}"

    print("GREEN CHECK: PASS")

verify_report()

GREEN CHECK: PASS


In [16]:
# Green-check verifier for Lab W3D3 (engine swap).
# Paste this as the last cell of your day-3 notebook and run it. It reads
# ab_report.json and checks the schema, that vLLM's concurrency-8 throughput
# beats Monday's batch-8 baseline, and that the speedup fields were computed.
#
# Last line is exactly one of:
#   GREEN CHECK: PASS
#   GREEN CHECK: FAIL (<reason>)
# No interactivity, no arguments; exit code matches.

import json, os


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def main() -> None:
    path = "ab_report.json"
    if not os.path.exists(path):
        fail(f"{path} not found; write it in Cell 5")
    try:
        with open(path) as fh:
            report = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"{path} is not valid JSON: {exc}")

    for key in ("baseline", "vllm", "speedup_by_concurrency"):
        if key not in report:
            fail(f"{path} missing key: {key}")

    baseline = report["baseline"]
    vllm = report["vllm"]
    speedup = report["speedup_by_concurrency"]

    if not isinstance(baseline, dict) or not baseline:
        fail("baseline must be a non-empty object (from Monday's baselines.json)")
    if not isinstance(vllm, dict) or not vllm:
        fail("vllm must be a non-empty object of measured throughput")
    if not isinstance(speedup, dict) or not speedup:
        fail("speedup_by_concurrency must be a non-empty object")

    # keys may be strings or ints depending on how the report was built; normalise
    def get_c(d, c):
        for k, v in d.items():
            if str(k) == str(c):
                return v
        return None

    base8 = get_c(baseline, 8)
    vllm8 = get_c(vllm, 8)
    if base8 is None:
        fail("baseline has no concurrency-8 (batch-8) number")
    if vllm8 is None:
        fail("vllm has no concurrency-8 number")
    if not isinstance(base8, (int, float)) or not isinstance(vllm8, (int, float)):
        fail("concurrency-8 throughput values must be numbers")

    # the headline claim of the day
    if not vllm8 > base8:
        fail(f"vllm concurrency-8 throughput ({vllm8}) not above baseline "
             f"batch-8 ({base8}); the engine swap should win here")

    # speedup fields must be computed (present and numeric for at least c=8)
    s8 = get_c(speedup, 8)
    if s8 is None or not isinstance(s8, (int, float)):
        fail("speedup_by_concurrency has no numeric value at concurrency 8")
    # sanity: the reported speedup should match vllm8/base8 within rounding
    expected = vllm8 / base8
    if abs(s8 - expected) > 0.1:
        fail(f"speedup at 8 ({s8}) does not match vllm/baseline "
             f"({expected:.2f}); recompute it")

    print(f"baseline batch-8: {base8}, vllm concurrency-8: {vllm8}")
    print(f"speedup at 8: {s8}x")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)


baseline batch-8: 96.6, vllm concurrency-8: 241.4
speedup at 8: 2.5x
GREEN CHECK: PASS
